In [ ]:
from pathlib import Path
from itertools import chain, repeat

import pandas as pd
from skimage.measure import regionprops_table
from skimage.io import imread
from nd2 import ND2File
import tifffile

def get_num_positions_nd2(in_file):
    """
    get the number of xy-positions / tiles in an nd2 file, will return 1 if file is just a single (potentially multichannel) stack
    """
    with ND2File(in_file) as reader:
        return reader.sizes['P'] if 'P' in reader.sizes else 1

In [ ]:
in_path = '/Users/josefin/PhD/weihua_UHRF1_CellCycle/'

segmentation_subdirectory = 'segmentation_nuclei_IllCorr'

images_subdirectory = 'images/2603011/tif_illumination_corrected'


out_subdirectory = 'region_properties_nuclei_IllCorr'

properties_to_include = ('label', 'area', 'intensity_mean', 'intensity_min', 'intensity_max', 'intensity_std')

In [ ]:
mask_files = sorted((Path(in_path) / segmentation_subdirectory).glob('[!.]*.tif'))

image_files = sorted((Path((in_path + images_subdirectory)).glob('*ch0.tif')))


# image_files_with_position = list(chain(*(zip(repeat(image_file), range(get_num_positions_nd2(image_file))) for image_file in image_files)))

In [ ]:
outdir = Path(in_path) / out_subdirectory

if not outdir.exists():
    outdir.mkdir()

for mask_file, image_file in zip(mask_files, image_files):

    mask = imread(mask_file)
    image = tifffile.imread(image_file)
    
    df = pd.DataFrame(regionprops_table(mask, properties=properties_to_include, intensity_image=image))

    # save image & mask file (relative to dataset folder)
    df['img_file_rprop'] = str(image_file.relative_to(Path(in_path + images_subdirectory)))
    df['mask_file'] = str(mask_file.relative_to(Path(in_path + segmentation_subdirectory)))
    # df['position_idx'] = position_idx

    outfile = outdir / (image_file.stem + '_regionprops.csv')
    df.to_csv(outfile, index=None)